<a href="https://colab.research.google.com/github/Innovatewithapple/bert-dense-retriever-benchmark/blob/main/EvaluateBeirBenchMark_QuoraDataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch datasets beir

In [ ]:
pip install rank-bm25

In [ ]:
import torch
from transformers import AutoTokenizer,AutoModel
from datasets import load_dataset
from beir.retrieval.evaluation import EvaluateRetrieval
from beir.retrieval.search.dense import DenseRetrievalExactSearch as DRES
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

In [ ]:
print("--- Loading Custom Model & Tokenizer ---")
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
import torch
from transformers import AutoTokenizer
from datasets import load_dataset
from beir.retrieval.evaluation import EvaluateRetrieval
from beir.retrieval.search.dense import DenseRetrievalExactSearch as DRES
from sentence_transformers import CrossEncoder

# --- 1. Import Your Local Classes ---
from configuration_bert_retriever import BertRetrieverConfig
from modeling_bert_retriever import BertRetriever

device = "cuda" if torch.cuda.is_available() else "cpu"

print("--- 1. Initializing Local Architecture & Checkpoint ---")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

config = BertRetrieverConfig(
    backbone_name="bert-base-uncased",
    hidden_size=768,
    dropout=0.2,
)

model = BertRetriever(config).to(device)
checkpoint = torch.load(
    "/content/best_model_epochj6_9693.pth",
    map_location=device,
    weights_only=False
)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# --- 2. Define the Custom Wrapper Class ---
# CHANGE 1: BeIR expects 'encode_passages' as the method name, not 'encode_corpus'
class CustomHFModelWrapper:
    def __init__(self, model, tokenizer, device):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device

    @torch.no_grad()
    def encode_queries(self, queries, batch_size=16, **kwargs):
        embeddings = []
        for i in range(0, len(queries), batch_size):
            batch = queries[i:i+batch_size]
            inputs = self.tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(self.device)
            out = self.model(**inputs)
            embeddings.append(out.cpu())
        return torch.cat(embeddings, dim=0).numpy()

    @torch.no_grad()
    def encode_corpus(self, passages, batch_size=16, **kwargs):
        text_passages = [doc["text"] for doc in passages]
        embeddings = []
        for i in range(0, len(text_passages), batch_size):
            batch = text_passages[i:i+batch_size]
            inputs = self.tokenizer(
                batch,
                padding=True,
                truncation=True,
                return_tensors="pt"
            ).to(self.device)
            out = self.model(**inputs)
            embeddings.append(out.cpu())
        return torch.cat(embeddings, dim=0).numpy()

print("\n--- 1. Streaming Quora Dataset from Hugging Face ---")
# Quora structures all configuration mappings directly under the "test" split
hf_corpus = load_dataset("BeIR/quora", "corpus", split="corpus")
hf_queries = load_dataset("BeIR/quora", "queries", split="queries")
hf_qrels = load_dataset("BeIR/quora-qrels", split="test")

print("--- 2. Filtering a Stable 5,000-Query Portfolio Slice ---")
# Pulling the first 1k unique query relations keeps loop execution around ~10 mins
sample_query_ids = set([str(row["query-id"]) for row in list(hf_qrels)])

queries = {str(row["_id"]): row["text"] for row in hf_queries if str(row["_id"]) in sample_query_ids}

qrels = {}
needed_corpus_ids = set()
for row in hf_qrels:
    q_id = str(row["query-id"])
    doc_id = str(row["corpus-id"])
    if q_id in sample_query_ids:
        if q_id not in qrels:
            qrels[q_id] = {}
        qrels[q_id][doc_id] = int(row["score"])
        needed_corpus_ids.add(doc_id)

corpus = {str(row["_id"]): {"text": row["text"], "title": row["title"]} for row in hf_corpus if str(row["_id"]) in needed_corpus_ids}
print(f"-> Loaded a validation pool of {len(queries)} queries and {len(corpus)} passages.")

# --- 3. Execute Pure Dense Retrieval & Cross-Encoder Reranking ---
wrapped_model = DRES(CustomHFModelWrapper(model, tokenizer, device), batch_size=16)

print("\n--- Phase 1: Running Base Vector Search (No BM25 Hybrid Needed) ---")
initial_results = wrapped_model.search(corpus, queries, top_k=100, score_function="cos_sim")

print("\n--- Phase 2: Running Cross-Encoder Re-ranking Loop ---")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=device)

reranked_results = {}
for query_id, doc_scores in initial_results.items():
    query_text = queries[query_id]
    pairs = [[query_text, corpus[doc_id]["text"]] for doc_id in doc_scores.keys()]
    scores = reranker.predict(pairs, batch_size=32, show_progress_bar=True)
    reranked_results[query_id] = {doc_id: float(score) for doc_id, score in zip(doc_scores.keys(), scores)}

print("\n--- Phase 3: Calculating Final Semantic Metrics ---")
retriever = EvaluateRetrieval(wrapped_model, score_function="cos_sim")
ndcg, _map, recall, precision = retriever.evaluate(qrels, reranked_results, retriever.k_values)

print("\n=============================================")
print("             QUORA TEST RESULTS              ")
print("=============================================")
print(f"NDCG@10:    {ndcg['NDCG@10']:.4f}")
print(f"Recall@10:  {recall['Recall@10']:.4f}")
print(f"Recall@100: {recall['Recall@100']:.4f}")
print("=============================================")

--- 1. Initializing Local Architecture & Checkpoint ---

--- 2. Streaming FULL TREC-COVID Dataset from Hugging Face ---
Loaded the ENTIRE TREC-COVID dataset: 50 queries and 35480 passages.

--- Phase 1A: Running Vector Search (Dense) ---


--- Phase 2: Merging Candidates & Cross-Encoder Reranking ---



--- Phase 3: Calculating Final Pipeline Metrics ---

             QUORA TEST RESULTS              
NDCG@10:    0.9686
Recall@10:  0.9858
Recall@100: 0.9938
